# Recursive partitioning across the shared 2D datasets

This cycles through a self-contained 2D dataset list. Each panel is a recursive partitioner driven by the named base classifier; multiclass backgrounds use probability-weighted class colors and entropy-based confidence fading.

In [ ]:
%config InlineBackend.figure_format = 'retina'
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

working_directory = Path.cwd()
project = next(
    (candidate for candidate in (working_directory, *working_directory.parents)
     if (candidate / 'recursive_partition').is_dir()),
    working_directory,
)
sys.path.insert(0, str(project))

from recursive_partition import (
    BaggedRecursivePartitionClassifier,
    RecursivePartitionClassifier,
    make_2d_dataset,
    plot_probability_heatmap,
)

In [ ]:
DATASETS = (
    'circles', 'moon', 'spirals', 'xor', 'checkerboard',
    'gaussian', 'blobs', 'anisotropic_blobs', 'classification',
)
DATA_PARAMETERS = {
    'n_samples': 1000, 'noise': 0.22, 'random_state': 42,
    'n_classes': 4, 'blob_center_radius': 3.0,
    'blob_cluster_standard_deviation': 1.5, 'circle_factor': 0.5,
    'spiral_turns': 2, 'spiral_noise': 0.03,
    'checkerboard_cells': 4, 'checkerboard_extent': 4.0,
    'anisotropy': 3.0, 'rotation': 0.35,
    'classification_class_sep': 1.0, 'classification_flip_y': 0.05,
}
BAGGING_N_ESTIMATORS = 30
BASE_MODELS = {
    'Linear SVM': RecursivePartitionClassifier(
        base_estimator=SVC(kernel='linear', C=1.0, class_weight='balanced')
    ),
    'Quadratic SVM': RecursivePartitionClassifier(
        base_estimator=SVC(kernel='poly', degree=2, coef0=1.0, gamma='scale', C=1.0, class_weight='balanced')
    ),
    'RBF SVM': RecursivePartitionClassifier(
        base_estimator=SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
    ),
    'QDA': RecursivePartitionClassifier(
        base_estimator=QuadraticDiscriminantAnalysis(reg_param=0.05)
    ),
}
MODELS = {
    **BASE_MODELS,
    **{
        f'{name} — bagged ({BAGGING_N_ESTIMATORS})': BaggedRecursivePartitionClassifier(
            estimator=clone(model),
            n_estimators=BAGGING_N_ESTIMATORS, n_jobs=-1, random_state=42,
        )
        for name, model in BASE_MODELS.items()
    },
}

def depth_summary(model):
    if hasattr(model, 'estimators_'):
        depths = [estimator.get_depth() for estimator in model.estimators_]
        return f'avg depth: {np.mean(depths):.1f}'
    return f'depth: {model.get_depth()}'

In [ ]:
for dataset in DATASETS:
    X, y = make_2d_dataset(dataset, **DATA_PARAMETERS)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )
    fig, axes = plt.subplots(4, 2, figsize=(14, 18), constrained_layout=True)
    print(dataset)
    for axis, (name, template) in zip(axes.ravel(), MODELS.items()):
        model = clone(template)
        model.fit(X_train, y_train)
        plot_probability_heatmap(
            model, X, y, X_train=X_train, y_train=y_train,
            title=f'Recursive {name} — accuracy: {model.score(X_test, y_test):.3f} — {depth_summary(model)}',
            ax=axis, colorbar=(axis is axes.ravel()[-1]),
        )
    fig.suptitle(dataset)
    plt.show()